# 🔴 Solution: Linear Self-Attention（面试版）

## 📋 题目描述

**难度：** Hard

实现基于核特征映射的线性自注意力。

线性注意力用特征映射 phi 替代 softmax，通过重排计算顺序将复杂度从 O(S^2*D) 降至 O(S*D^2)。

**签名:** `linear_attention(Q, K, V) -> Tensor`

**参数:**
- `Q` — 查询张量 (B, S, D_k)
- `K` — 键张量 (B, S, D_k)
- `V` — 值张量 (B, S, D_v)

**返回:** 注意力输出 (B, S, D_v)

**约束:**
- 特征映射：`phi(x) = elu(x) + 1`
- 计算 `phi(Q) @ (phi(K)^T @ V)` 而非 `softmax(Q @ K^T) @ V`
- 通过 `phi(Q) @ sum(phi(K))` 归一化（加 `eps=1e-6` 保证数值稳定）

**提示：** `phi(x) = elu(x) + 1`。输出 = `(phi(Q) @ (phi(K).T @ V)) / (phi(Q) @ phi(K).sum(dim=-2, keepdim=True).T)`


In [ ]:
import torch
import math

In [ ]:
# ✅ INTERVIEW

def linear_attention(query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
    d_k = key.size(-1)

    # ---- Step 1: 核函数映射 ----
    # elu(x) + 1：近似 exp，但计算更简单
    # 关键：保持非负性，这是线性注意力能分解的前提
    Q = torch.nn.functional.elu(query) + 1
    K = torch.nn.functional.elu(key) + 1

    # ---- Step 2: 先算 K^T @ V ----
    # 标准注意力顺序：(Q @ K^T) @ V → O(S²D)
    # 线性注意力顺序：Q @ (K^T @ V) → O(SD²)
    # 利用矩阵乘法结合律，改变计算顺序！
    # K^T: [B, H, D, S] @ V: [B, H, S, D] → [B, H, D, D]
    KV = K.transpose(-2, -1) @ value

    # ---- Step 3: Q @ KV ----
    # [B, H, S, D] @ [B, H, D, D] → [B, H, S, D]
    output = Q @ KV

    # ---- Step 4: 归一化 ----
    # Z = Σ φ(k_j)，类似 softmax 的分母
    # 每个 query 的输出需要除以其对应的 Z
    Z = Q @ K.transpose(-2, -1).sum(dim=-1, keepdim=True)
    output = output / (Z + 1e-6)

    return output

# 面试追问：
# Q: 为什么叫"线性"注意力？
# A: 复杂度与序列长度成线性关系 O(SD²)，而非 O(S²)
# Q: 为什么用 elu+1？
# A: 需要非负核函数来近似 softmax，elu+1 保证非负
# Q: 缺点？
# A: 核函数近似损失精度，实际效果可能不如标准注意力

In [ ]:
Q=torch.randn(1,8,16); K=torch.randn(1,8,16); V=torch.randn(1,8,32)
print('Shape:', linear_attention(Q,K,V).shape)

## 📝 核心思路总结

1. **核心思想**：改变矩阵乘法顺序，O(S²D) → O(SD²)
2. **核函数**：elu+1 近似 exp，保持非负性
3. **结合律**：Q(K^TV) 先算 KV 再乘 Q
4. **归一化**：除以 Z 保证数值合理性